# D-Fire se Fire & Smoke YOLO Training
Colab me **Runtime > Change runtime type > T4 GPU** select karo, phir sab cells upar se neeche chalao. Dataset D-Fire hai (CC BY 4.0); attribution neeche di gayi hai.

In [ ]:
!pip install -q ultralytics huggingface_hub
!nvidia-smi

## 1. D-Fire dataset download
Hugging Face par D-Fire ka raw mirror download hoga (~3.2 GB compressed; extraction ke liye extra disk space chahiye). Original D-Fire license CC BY 4.0 hai. Required citation: Pedro Vinícius Almeida Borges de Venâncio, Adriano Chaves Lisboa, Adriano Vilela Barbosa, “An automatic fire detection system based on deep convolutional neural networks for low-power, resource-constrained devices,” Neural Computing and Applications, 2022.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import tarfile

REPO = "hanvithSai/gavin-dfire-raw"  # raw D-Fire mirror; original dataset attribution applies
RAW = Path("/content/dfire_raw")
RAW.mkdir(parents=True, exist_ok=True)
for split in ("train", "test"):
    archive = hf_hub_download(repo_id=REPO, repo_type="dataset", filename=f"{split}.tar")
    with tarfile.open(archive) as tar:
        tar.extractall(RAW, filter="data")
print("Extracted D-Fire to", RAW)

## 2. Official split banao aur YOLO data.yaml likho
D-Fire mirror me 17,221 train aur 4,306 test images hain. Training split me se fixed seed ke saath 3,099 validation images alag karenge: final counts 14,122 train, 3,099 val, 4,306 test.

In [ ]:
import random, yaml

def find_split_dirs(split):
    image_dirs = [p for p in RAW.rglob("images") if p.parent.name == split]
    if len(image_dirs) != 1:
        raise RuntimeError(f"Expected one {split}/images folder, found: {image_dirs}")
    images = image_dirs[0]
    labels = images.parent / "labels"
    if not labels.is_dir():
        raise FileNotFoundError(f"Missing labels folder: {labels}")
    return images, labels

train_images, train_labels = find_split_dirs("train")
test_images, test_labels = find_split_dirs("test")
image_exts = {".jpg", ".jpeg", ".png"}
all_train = sorted(p for p in train_images.iterdir() if p.suffix.lower() in image_exts)
all_test = sorted(p for p in test_images.iterdir() if p.suffix.lower() in image_exts)
assert len(all_train) == 17221, f"Expected 17,221 train images, found {len(all_train)}"
assert len(all_test) == 4306, f"Expected 4,306 test images, found {len(all_test)}"

val_set = set(random.Random(2026).sample(all_train, 3099))
OUT = Path("/content/dfire_yolo")
for split in ("train", "val", "test"):
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

def link_pair(image, labels_dir, split):
    label = labels_dir / f"{image.stem}.txt"
    if not label.is_file():
        raise FileNotFoundError(f"Missing annotation for {image.name}: {label}")
    (OUT / "images" / split / image.name).symlink_to(image)
    (OUT / "labels" / split / label.name).symlink_to(label)

for image in all_train:
    link_pair(image, train_labels, "val" if image in val_set else "train")
for image in all_test:
    link_pair(image, test_labels, "test")

assert len(list((OUT / "images/train").iterdir())) == 14122
assert len(list((OUT / "images/val").iterdir())) == 3099
assert len(list((OUT / "images/test").iterdir())) == 4306
cfg = {"path": str(OUT), "train": "images/train", "val": "images/val",
       "test": "images/test", "names": ["smoke", "fire"]}
yaml_path = Path("/content/data.yaml")
yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml_path.read_text())

## 3. YOLOv8s train karo (T4 GPU recommended; time Colab GPU par depend karega)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")
model.train(data="/content/data.yaml", epochs=60, imgsz=640, batch=16,
            patience=15, project="/content/runs", name="dfire", exist_ok=True)

## 4. Result check

In [ ]:
best = YOLO("/content/runs/dfire/weights/best.pt")
val_metrics = best.val(data="/content/data.yaml", split="val")
test_metrics = best.val(data="/content/data.yaml", split="test")
print("Validation mAP50:", val_metrics.box.map50)
print("Test mAP50:", test_metrics.box.map50)

## 5. Trained model download karke project folder me rakho
Downloaded `best.pt` ko `fire_smoke_ai` project folder me copy karo. `config.py` ka `DETECTOR = "auto"` file milte hi YOLO use karega.

In [ ]:
from google.colab import files
files.download("/content/runs/dfire/weights/best.pt")